In [1]:
import pandas as pd

In [2]:
sidata = pd.read_csv("/workspaces/service-data/outputs/snapshots/2025-03-01/utils/si_all.csv", sep=";")

In [4]:
si23 = sidata[sidata["fiscal_yr"] == "2023-2024"].copy()
si22 = sidata[sidata["fiscal_yr"] == "2022-2023"].copy()
siEnd22 = sidata.query("fiscal_yr in ['2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023']").copy()

In [5]:
si23_ext = si23[si23["service_scope_ext_or_ent"]]
si22_ext = si22[si22["service_scope_ext_or_ent"]]

In [6]:
print(f"In 2023-24, there are {si23.shape[0]} services reported, of which {si23_ext.shape[0]} are extern/enterprise, and {si22_ext.shape[0]} services in 2022-23.")

In 2023-24, there are 1715 services reported, of which 1681 are extern/enterprise, and 1690 services in 2022-23.


In [7]:
si_cols = [
    'fiscal_yr', 'service_id', 'service_name_en', 'service_name_fr', 'service_description_en', 'service_description_fr', 'service_type',
    'service_recipient_type', 'service_scope', 'client_target_groups', 'program_name_en', 'program_name_fr', 'client_feedback_channel',
    'service_fee', 'os_account_registration', 'os_authentication', 'os_application', 'os_decision', 'os_issuance', 'os_issue_resolution_feedback',
    'os_comments_client_interaction_en', 'os_comments_client_interaction_fr', 'last_service_review', 'last_service_improvement', 'sin_usage',
    'cra_bn_identifier_usage', 'num_phone_enquiries', 'num_applications_by_phone', 'num_website_visits', 'num_applications_online',
    'num_applications_in_person', 'num_applications_by_mail', 'num_applications_by_email', 'num_applications_by_fax', 'num_applications_by_other',
    'special_remarks_en', 'special_remarks_fr', 'service_uri_en', 'service_uri_fr', 'num_applications_total', 'org_name_variant',
    'org_id', 'department_en', 'department_fr', 'program_id', 'automated_decision_system_description_fr', 'automated_decision_system',
    'automated_decision_system_description_en', 'service_scope_ext_or_ent', 'fy_org_id_service_id'    
]

In [8]:
si23_ext = si23_ext.filter(si_cols).copy()

In [9]:
prog_cols = {"fiscal_yr":"string", "service_id":"string", "service_name_en":"string", "program_id":"string", "program_name_en":"string", "org_name_variant":"string", "org_id":"int", "department_en":"string"}

In [10]:
org_names = list(set(si23_ext["org_name_variant"].values))
Nlines = int(len(org_names) / 10)
for k in range(Nlines):
    if k*10 < len(org_names):
        print(", ".join(org_names[k*10:k*10+10]))
print(", ".join(org_names[(Nlines)*10:]))

pmprb-cepmb, csec-cstc, lac-bac, acoa-apeca, phac-aspc, cta-otc, sshrc-crsh, ic, feddevontario, cgc-ccg
ssc-spc, csps-efpc, statcan, cihr-irsc, pc, nsira-ossnr, cannor, cer-rec, tbs-sct, iaac-aeic
tc, fintrac-canafe, irb-cisr, csc-scc, ppsc-sppc, hc-sc, dfo-mpo, ccohs-cchst, pwgsc-tpsgc, fin
esdc-edsc, fcac-acfc, cas-satj, nrcan-rncan, wage, isc-sac, jus, csa-asc, cfia-acia, aandc-aadnc
ced-dec, oci-bec, nserc-crsng, opc-cpvp, rcmp-grc, pco-bcp, pbc-clcc, cbsa-asfc, aafc-aac, cpc-cpp
chrc-ccdp, oag-bvg, cics-scic, crtc, nbc-ccbn, atssc-scdata, infc, polar-polaire, pch, psc-cfp
vac-acc, fja-cmf, dnd-mdn, nfb-onf, mpcc-cppm, cra-arc, vrab-tacra, cic, ec, pacifican
dfatd-maecd, nrc-cnrc, fednor, osfi-bsif, fpcc-cpac, prairiescan, ps-sp, cb-cda


In [11]:
org_ids = list(set(si23_ext["org_id"].values))
Nlines = int(len(org_ids) / 10)
for k in range(Nlines):
    if k*10 < len(org_ids):
        print(" ".join([f"{i:.0f}" for i in org_ids[k*10:k*10+10]]))
print(" ".join([f"{i:.0f}" for i in org_ids[(Nlines)*10:]]))

1 12 26 539 552 46 47 560 561 55
63 65 69 71 74 76 86 93 95 99
110 114 117 118 122 123 124 125 126 127
128 129 130 132 133 134 135 136 137 138
139 140 141 150 151 152 174 199 210 218
221 222 223 227 228 230 237 238 240 246
247 248 253 256 263 266 278 280 282 295
297 302 305 306 313 326 333 348


In [12]:
si_prog = si23_ext.filter(list(prog_cols.keys())).copy()
si_prog.shape

(1681, 8)

In [14]:
# Change the data type of columns
si_prog = si_prog.astype(prog_cols)

In [15]:
#si_prog[:4]

In [17]:
# Get the one porgram ID per row as a service can have many programs
si_prog["prog_id_split"] = si_prog["program_id"].str.split(",")
si_prog = si_prog.explode("prog_id_split")
si_prog.shape

(2249, 9)

In [37]:
# Compute the unique department-programID identifiers
si_prog["org-prog"] = si_prog["org_id"].astype(str) + "-" + si_prog["prog_id_split"]
#si_prog[:5]

In [19]:
# Load the spending data
spending_data = pd.read_csv("/workspaces/service-data/notebooks/svc-spending/gcInfobase_expenditure.csv")

In [20]:
spending_data.head(4)

,fy_ef,organization_id,organization,core_responsibility,program_id,program_name,planned_spending_1,actual_spending,planned_spending_2,planned_spending_3,planned_ftes_1,actual_ftes,planned_ftes_2,planned_ftes_3,planning_explanation,variance_explanation
0,FY 2018-19,1,Department of Agriculture and Agri-Food,Domestic and International Markets,BWN01,Trade and Market Expansion,53105701.0,52360723.82,53014508.0,53014508.0,171.0,183.0,171.0,171.0,NaN,NaN
1,FY 2018-19,1,Department of Agriculture and Agri-Food,Domestic and International Markets,BWN02,Sector Engagement and Development,33331249.0,34247456.65,30455570.0,30455570.0,179.0,184.0,179.0,179.0,NaN,NaN
2,FY 2018-19,1,Department of Agriculture and Agri-Food,Domestic and International Markets,BWN03,Farm Products Council of Canada,3048578.0,2520779.52,3048552.0,3048552.0,23.0,17.0,23.0,23.0,NaN,Actual spending was lower than planned spendin...
3,FY 2018-19,1,Department of Agriculture and Agri-Food,Domestic and International Markets,BWN04,Dairy Programs,94238832.0,99881679.91,83258832.0,78288832.0,36.0,46.0,37.0,37.0,NaN,Actual full-time equivalents were higher than ...


In [25]:
spending_columns = {
    "fy_ef":"string",
    "organization_id":"int",
    "organization":"string",
    "core_responsibility":"string",
    "program_id":"string",
    "program_name":"string",
    "planned_spending_1":"float",
    "planned_spending_2":"float",
    "planned_spending_3":"float",
    "actual_spending":"float",
    "planned_ftes_1":"float",
    "planned_ftes_2":"float",
    "planned_ftes_3":"float",
    "actual_ftes":"float",
}

In [28]:
spending_data_cleaned = spending_data.filter(list(spending_columns.keys())).copy()
spending_data_cleaned["fy_ef"] = spending_data_cleaned["fy_ef"].str.removeprefix("FY ")

In [32]:
spending_23 = spending_data_cleaned.query("fy_ef == '2023-24'")
spending_24 = spending_data_cleaned.query("fy_ef == '2024-25'")

In [33]:
spending_filter_cols = ["fy_ef", "organization_id", "organization", "program_id", "program_name"]

In [34]:
spending_24_prog = spending_24.filter(spending_filter_cols + ["planned_spending_2"])
spending_24_prog["org-prog"] = spending_24_prog["organization_id"].astype(str) + "-" + spending_24_prog["program_id"]
spending_24_prog.shape

(1180, 7)

In [36]:
spending_24_prog.tail(5)

,fy_ef,organization_id,organization,program_id,program_name,planned_spending_2,org-prog
8503,2024-25,560,Pacific Economic Development Agency of Canada,ISS10,Internal Services,7267057.0,560-ISS10
8504,2024-25,561,Federal Economic Development Agency for Northe...,BYL01,Business Development,26006699.0,561-BYL01
8505,2024-25,561,Federal Economic Development Agency for Northe...,BYL02,Regional Innovation Ecosystem,7695956.0,561-BYL02
8506,2024-25,561,Federal Economic Development Agency for Northe...,BYL03,Community Economic Development and Diversifica...,21716635.0,561-BYL03
8507,2024-25,561,Federal Economic Development Agency for Northe...,ISS00,Internal Services,4581144.0,561-ISS00


In [40]:
# Difference between SI program IDs and spending programs IDS, indicating incorrect program tagged to a service
set_si23 = set(si_prog["org-prog"].values)
set_sp24 = set(spending_24_prog["org-prog"].values)
si_progNotInSpending = set_si23 - set_sp24
sp_progNotInSI = set_sp24 - set_si23

In [42]:
len(si_progNotInSpending)

75

In [47]:
# spending for 2023-24
spending_23_prog = spending_23.filter(spending_filter_cols + ["planned_spending_3"])
spending_23_prog["org-prog"] = spending_23_prog["organization_id"].astype(str) + "-" + spending_23_prog["program_id"]
spending_23_prog.shape

(1173, 7)

In [48]:
set_sp23 = set(spending_23_prog["org-prog"].values)

In [49]:
tmp1 = si_progNotInSpending.intersection(set_sp23)

In [50]:
len(tmp1)

6

In [51]:
tmp1

{'134-BTO06', '134-BTO07', '137-BGS02', '228-BNQ13', '228-BNQ16', '228-BNQ23'}

In [54]:
spending_23_prog[spending_23_prog["org-prog"].str.contains(",")].shape

(0, 7)